# 学術データベースのアクセス検査

対象 DOI を 1 件だけ照会し、Crossref・Semantic Scholar の公開 API と、Scopus・Web of Science の API 利用可否を区別して確認する。大量取得、Web 画面の自動操作、機関認証の自動化は行わない。

- Crossref は認証なしで照会する。Semantic Scholar も公開 API だが、共有環境でレート制限されたときだけ `SEMANTIC_SCHOLAR_API_KEY` をローカル環境変数から渡せる。
- Scopus は `SCOPUS_API_KEY`、Web of Science Starter API は `WOS_API_KEY` がローカル環境変数にある場合だけ照会する。値はノートブック、出力ファイル、実行ログに保存しない。
- 大阪大学の個人 ID・パスワードは使わない。図書館経由の Web UI 利用権と API キーの発行・契約は別の確認事項である。

公式資料: [Crossref REST API](https://www.crossref.org/documentation/retrieve-metadata/rest-api/)、[Semantic Scholar API](https://www.semanticscholar.org/product/api)、[Scopus APIs](https://dev.elsevier.com/sc_apis.html)、[Web of Science Starter API](https://developer.clarivate.com/apis/wos-starter)。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlencode, urlparse
from urllib.request import HTTPRedirectHandler, Request, build_opener, urlopen
import json
import os

import pandas as pd

# 実行場所に依存せず、リポジトリ直下を探索する。
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'access_check'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOI = '10.1126/science.aaf5239'  # Sinatra et al. (2016), Random Impact Rule
USER_AGENT = 'Sci-sci-access-check/1.0 (single-record metadata validation)'
RUN_AT_UTC = datetime.now(timezone.utc).isoformat()

print(f'対象 DOI: {DOI}')
print(f'出力先: {OUTPUT_DIR.relative_to(ROOT)}')

対象 DOI: 10.1126/science.aaf5239
出力先: data/derived/access_check


## 安全な認証の扱い

このノートブックは大阪大学 ID・パスワードを受け取らず、保存もしない。Scopus または WoS の API 利用権が確認でき、発行済み API キーがある場合に限り、Jupyter を起動するローカル端末で次のように設定する。

```sh
export SEMANTIC_SCHOLAR_API_KEY='発行済みのキー（429 時のみ必要）'
export SCOPUS_API_KEY='発行済みのキー'
export WOS_API_KEY='発行済みのキー'
```

Web of Science / Scopus の Web UI 用の ID・パスワードは、これらの API キーではない。未設定なら該当 API は `not_attempted` として記録される。

In [2]:
def request_json(source, url, headers=None):
    """
    1 件の API 応答だけを取得し、認証情報を含まない最小限の実行結果を返す。
    HTTP エラーも例外で止めず、利用権・レート制限・一時的障害を区別できる形にする。
    """
    request_headers = {'Accept': 'application/json', 'User-Agent': USER_AGENT}
    if headers:
        request_headers.update(headers)

    started = perf_counter()
    try:
        request = Request(url, headers=request_headers)
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode('utf-8'))
            return {
                'source': source,
                'attempted': True,
                'http_status': response.status,
                'ok': True,
                'duration_ms': round((perf_counter() - started) * 1000),
                'note': 'JSON を取得できた。',
            }, payload
    except HTTPError as error:
        # 本文は保存せず、HTTP ステータスだけで認証・権限上の問題を判定する。
        if error.code == 429:
            note = 'HTTP 429: レート制限。時間を置くか、発行済みの API キーを使う。'
        elif error.code in (401, 403):
            note = f'HTTP {error.code}: API キー、契約、またはネットワーク上の権限を確認する。'
        else:
            note = f'HTTP {error.code}: {error.reason}'
        return {
            'source': source,
            'attempted': True,
            'http_status': error.code,
            'ok': False,
            'duration_ms': round((perf_counter() - started) * 1000),
            'note': note,
        }, None
    except (URLError, TimeoutError, json.JSONDecodeError) as error:
        return {
            'source': source,
            'attempted': True,
            'http_status': None,
            'ok': False,
            'duration_ms': round((perf_counter() - started) * 1000),
            'note': f'{type(error).__name__}: {error}',
        }, None


def skipped_result(source, environment_variable):
    return {
        'source': source,
        'attempted': False,
        'http_status': None,
        'ok': False,
        'duration_ms': 0,
        'note': f'{environment_variable} が未設定のため、API を照会していない。',
    }, None


class NoRedirect(HTTPRedirectHandler):
    """SAMLの認証画面へ進まず、最初のリダイレクトだけを安全に記録する。"""

    def redirect_request(self, request, file_pointer, status_code, message, headers, new_url):
        return None


def request_initial_redirect(source, url):
    """
    大阪大学学外アクセスの入口が認証基盤へ到達するかを確認する。
    Locationのクエリ文字列、Cookie、認証情報は読み出し・保存しない。
    """
    started = perf_counter()
    try:
        opener = build_opener(NoRedirect)
        request = Request(url, headers={'User-Agent': USER_AGENT})
        with opener.open(request, timeout=30) as response:
            return {
                'source': source,
                'attempted': True,
                'http_status': response.status,
                'ok': response.status == 200,
                'duration_ms': round((perf_counter() - started) * 1000),
                'note': '認証前の入口が HTTP 200 を返した。',
            }, None
    except HTTPError as error:
        if error.code in (301, 302, 303, 307, 308):
            redirect_host = urlparse(error.headers.get('Location', '')).hostname
            return {
                'source': source,
                'attempted': True,
                'http_status': error.code,
                'ok': redirect_host == 'ou-idp.auth.osaka-u.ac.jp',
                'duration_ms': round((perf_counter() - started) * 1000),
                'note': f'認証前の入口に到達。リダイレクト先ホスト: {redirect_host}（SAML認証は実行しない）。',
            }, None
        return {
            'source': source,
            'attempted': True,
            'http_status': error.code,
            'ok': False,
            'duration_ms': round((perf_counter() - started) * 1000),
            'note': f'HTTP {error.code}: {error.reason}',
        }, None
    except (URLError, TimeoutError) as error:
        return {
            'source': source,
            'attempted': True,
            'http_status': None,
            'ok': False,
            'duration_ms': round((perf_counter() - started) * 1000),
            'note': f'{type(error).__name__}: {error}',
        }, None

In [3]:
encoded_doi = quote(DOI, safe='')

crossref_url = f'https://api.crossref.org/works/{encoded_doi}'
semantic_scholar_url = (
    f'https://api.semanticscholar.org/graph/v1/paper/DOI:{encoded_doi}?'
    + urlencode({'fields': 'title,year,citationCount,authors'})
)

results = []
payloads = {}

result, payload = request_json('Crossref', crossref_url)
results.append(result)
payloads['Crossref'] = payload

semantic_scholar_key = os.environ.get('SEMANTIC_SCHOLAR_API_KEY')
semantic_headers = {'x-api-key': semantic_scholar_key} if semantic_scholar_key else None
result, payload = request_json('Semantic Scholar', semantic_scholar_url, semantic_headers)
results.append(result)
payloads['Semantic Scholar'] = payload

pd.DataFrame(results)

,source,attempted,http_status,ok,duration_ms,note
0,Crossref,True,200,True,826,JSON を取得できた。
1,Semantic Scholar,True,429,False,255,HTTP 429: レート制限。時間を置くか、発行済みの API キーを使う。


## 大阪大学学外アクセス入口の確認

下の検査は Scopus と WoS の学外アクセス入口に1回だけ接続し、最初の HTTP リダイレクトで停止する。SAML 認証画面には進まず、個人ID・パスワード・Cookie・リダイレクト先URLのクエリ文字列を取得・保存しない。

In [4]:
uosaka_ui_entrypoints = {
    'Scopus Web UI（大阪大学学外アクセス）': (
        'https://osaka-u.idm.oclc.org/login?url=https%3A%2F%2Fwww.scopus.com%2F'
    ),
    'Web of Science Web UI（大阪大学学外アクセス）': (
        'https://osaka-u.idm.oclc.org/login?url=https%3A%2F%2Fwww.webofscience.com%2F'
    ),
}

for source, url in uosaka_ui_entrypoints.items():
    result, payload = request_initial_redirect(source, url)
    results.append(result)
    payloads[source] = payload

pd.DataFrame(results)

,source,attempted,http_status,ok,duration_ms,note
0,Crossref,True,200,True,826,JSON を取得できた。
1,Semantic Scholar,True,429,False,255,HTTP 429: レート制限。時間を置くか、発行済みの API キーを使う。
2,Scopus Web UI（大阪大学学外アクセス）,True,302,True,385,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...
3,Web of Science Web UI（大阪大学学外アクセス）,True,302,True,384,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...


In [5]:
# API キーは HTTP ヘッダーにのみ渡し、URL・出力・保存データには含めない。
scopus_key = os.environ.get('SCOPUS_API_KEY')
if scopus_key:
    scopus_url = 'https://api.elsevier.com/content/search/scopus?' + urlencode({
        'query': f'DOI({DOI})',
        'count': 1,
        'field': 'dc:identifier,prism:doi,citedby-count',
    })
    result, payload = request_json('Scopus', scopus_url, {'X-ELS-APIKey': scopus_key})
else:
    result, payload = skipped_result('Scopus', 'SCOPUS_API_KEY')
results.append(result)
payloads['Scopus'] = payload

wos_key = os.environ.get('WOS_API_KEY')
if wos_key:
    wos_url = 'https://api.clarivate.com/apis/wos-starter/v1/documents?' + urlencode({
        'db': 'WOS',
        'q': f'DO=({DOI})',
        'limit': 1,
        'page': 1,
        'detail': 'short',
    })
    result, payload = request_json('Web of Science Starter API', wos_url, {'X-ApiKey': wos_key})
else:
    result, payload = skipped_result('Web of Science Starter API', 'WOS_API_KEY')
results.append(result)
payloads['Web of Science Starter API'] = payload

pd.DataFrame(results)

,source,attempted,http_status,ok,duration_ms,note
0,Crossref,True,200.0,True,826,JSON を取得できた。
1,Semantic Scholar,True,429.0,False,255,HTTP 429: レート制限。時間を置くか、発行済みの API キーを使う。
2,Scopus Web UI（大阪大学学外アクセス）,True,302.0,True,385,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...
3,Web of Science Web UI（大阪大学学外アクセス）,True,302.0,True,384,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...
4,Scopus,False,NaN,False,0,SCOPUS_API_KEY が未設定のため、API を照会していない。
5,Web of Science Starter API,False,NaN,False,0,WOS_API_KEY が未設定のため、API を照会していない。


In [6]:
def selected_metadata(source, payload):
    """API ごとに異なる応答から、比較に必要な最小限の項目だけを抽出する。"""
    if not payload:
        return {'title': None, 'year': None, 'citation_count': None, 'result_count': None}

    if source == 'Crossref':
        work = payload.get('message', {})
        date_parts = work.get('published', work.get('issued', {})).get('date-parts', [[None]])
        return {
            'title': (work.get('title') or [None])[0],
            'year': date_parts[0][0] if date_parts and date_parts[0] else None,
            'citation_count': work.get('is-referenced-by-count'),
            'result_count': 1,
        }

    if source == 'Semantic Scholar':
        return {
            'title': payload.get('title'),
            'year': payload.get('year'),
            'citation_count': payload.get('citationCount'),
            'result_count': 1,
        }

    if source == 'Scopus':
        envelope = payload.get('search-results', {})
        entries = envelope.get('entry', [])
        first = entries[0] if entries else {}
        return {
            'title': first.get('dc:title'),
            'year': first.get('prism:coverDate', '')[:4] or None,
            'citation_count': first.get('citedby-count'),
            'result_count': envelope.get('opensearch:totalResults'),
        }

    if source == 'Web of Science Starter API':
        metadata = payload.get('metadata', {})
        hits = payload.get('hits', [])
        first = hits[0] if hits else {}
        return {
            'title': first.get('title'),
            'year': first.get('source', {}).get('publishYear'),
            'citation_count': first.get('citations', {}).get('count'),
            'result_count': metadata.get('total'),
        }

    return {'title': None, 'year': None, 'citation_count': None, 'result_count': None}

rows = []
for result in results:
    row = {
        'source': result['source'],
        'attempted': result['attempted'],
        'http_status': result['http_status'],
        'ok': result['ok'],
        'duration_ms': result['duration_ms'],
        'note': result['note'],
        'doi': DOI,
    }
    row.update(selected_metadata(result['source'], payloads[result['source']]))
    rows.append(row)

summary = pd.DataFrame(rows)
summary

,source,attempted,http_status,ok,duration_ms,note,doi,title,year,citation_count,result_count
0,Crossref,True,200.0,True,826,JSON を取得できた。,10.1126/science.aaf5239,Quantifying the evolution of individual scient...,2016.0,499.0,1.0
1,Semantic Scholar,True,429.0,False,255,HTTP 429: レート制限。時間を置くか、発行済みの API キーを使う。,10.1126/science.aaf5239,None,NaN,NaN,NaN
2,Scopus Web UI（大阪大学学外アクセス）,True,302.0,True,385,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...,10.1126/science.aaf5239,None,NaN,NaN,NaN
3,Web of Science Web UI（大阪大学学外アクセス）,True,302.0,True,384,認証前の入口に到達。リダイレクト先ホスト: ou-idp.auth.osaka-u.ac.j...,10.1126/science.aaf5239,None,NaN,NaN,NaN
4,Scopus,False,NaN,False,0,SCOPUS_API_KEY が未設定のため、API を照会していない。,10.1126/science.aaf5239,None,NaN,NaN,NaN
5,Web of Science Starter API,False,NaN,False,0,WOS_API_KEY が未設定のため、API を照会していない。,10.1126/science.aaf5239,None,NaN,NaN,NaN


In [7]:
# 応答全文ではなく、上で抽出した比較用メタデータと実行状態だけを保存する。
csv_path = OUTPUT_DIR / 'access_check_summary.csv'
json_path = OUTPUT_DIR / 'access_check_summary.json'

summary.to_csv(csv_path, index=False)
clean_rows = summary.where(pd.notna(summary), None).to_dict(orient='records')
with json_path.open('w', encoding='utf-8') as handle:
    json.dump(
        {
            'run_at_utc': RUN_AT_UTC,
            'doi': DOI,
            'scope': 'single-record API access check; no Web UI automation or bulk download',
            'results': clean_rows,
        },
        handle,
        ensure_ascii=False,
        indent=2,
    )

print(f'保存: {csv_path.relative_to(ROOT)}')
print(f'保存: {json_path.relative_to(ROOT)}')

保存: data/derived/access_check/access_check_summary.csv
保存: data/derived/access_check/access_check_summary.json


## 判定の読み方

- Crossref / Semantic Scholar の `200` は、この実行環境から公開メタデータ API に到達できたことを示す。Semantic Scholar の `429` は共有IP等でのレート制限であり、収録・利用権がないという意味ではない。引用数の値がデータベース間で一致することは期待しない。
- Scopus / WoS の `200` は、設定済み API キーとその時点の契約・IP 等の権限で対象 API に到達できたことを示す。`401` / `403` は通常、キー、契約、またはネットワーク上の権限を追加確認すべき状態である。
- `not_attempted` は Web UI の利用不可を意味しない。キーが未設定なので API を意図的に呼ばなかっただけである。
- 大阪大学の Web UI は、認証前の入口と SAML リダイレクトまでをこのノートブックで確認する。個人認証は秘密情報を必要とするため実行・保存しない。成功時も、機械的な大量ダウンロードはしない。